In [1]:
import openai

client = openai.OpenAI()

# 요구사항 2: LLM에게 함수를 설명하는 프롬프트를 작성
SYSTEM_PROMPT="""
너는 영화 전문 Agent야.
다음과 같은 함수를 가지고 있어.

`get_popular_movies()` - /movies에서 인기 영화를 가져옵니다.
`get_movie_details(id)` - /movies/:id에서 영화 정보를 가져옵니다.
`get_movie_credits(id)` - /movies/:id/credits에서 출연진 및 제작진을 가져옵니다.

사용자 요청에 맞게 알맞은 함수의 이름과 인자를 알려줘.

다음과 같은 json 형태로 출력해줘.
{"function" : "<함수명>", "arguments":{"id":<인자 값>}}

"""
# 요구사항 4 : 다음과 같은 입력으로 테스트하세요
TEST_INPUT = ["지금 인기 있는 영화가 무엇인지 알려줘", "movie ID 550에 해당하는 영화가 무엇인지 알려줘", "movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘"]

for user_input in TEST_INPUT:    

    response = client.chat.completions.create(
        model="gpt-4o-mini", # 요구사항 1: 모델은 gpt-4o-mini를 사용
        messages=[
            {"role":"system","content":SYSTEM_PROMPT},
            {"role":"user","content":user_input}
            ]
    )

    # 요구사항 3: 모델이 함수명과 인자(arguments)를 올바르게 출력하는 것을 확인
    print(f"Input : {user_input}")
    print(f"Response : {response.choices[0].message.content}")
    print()

Input : 지금 인기 있는 영화가 무엇인지 알려줘
Response : {"function" : "get_popular_movies", "arguments":{}}

Input : movie ID 550에 해당하는 영화가 무엇인지 알려줘
Response : {"function" : "get_movie_details", "arguments":{"id":550}}

Input : movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘
Response : {"function" : "get_movie_credits", "arguments":{"id":550}}



API 요청

In [2]:
import requests, json

BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"

def get_popular_movies():
    return requests.get(f"{BASE_URL}/movies").json()

def get_movie_details(id):
    return requests.get(f"{BASE_URL}/movies/{id}").json()

def get_movie_credits(id):
    return requests.get(f"{BASE_URL}/movies/{id}/credits").json()


FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits
}

In [3]:

for user_input in TEST_INPUT:    

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role":"system","content":SYSTEM_PROMPT},
            {"role":"user","content":user_input}
            ]
    )
    agent_response = json.loads(response.choices[0].message.content)
    fn_name = agent_response["function"]
    arg_value = agent_response["arguments"]
    api_response = FUNCTION_MAP[fn_name](**arg_value)
    
    print(f"Input : {user_input}")
    print(f"Agent Response : {agent_response}")
    print(f"API Response :{api_response}")
    print()

Input : 지금 인기 있는 영화가 무엇인지 알려줘
Agent Response : {'function': 'get_popular_movies', 'arguments': {}}
API Response :[{'adult': False, 'backdrop_path': 'https://image.tmdb.org/t/p/w1280/4k99kV4R1bbbrsnjR205v91Xbin.jpg', 'genre_ids': [27], 'id': 1339713, 'title': 'Obsession', 'original_language': 'en', 'original_title': 'Obsession', 'overview': 'After breaking the mysterious "One Wish Willow" to win his crush\'s heart, a hopeless romantic finds himself getting exactly what he asked for but soon discovers that some desires come at a dark, sinister price.', 'popularity': 898.7716, 'poster_path': 'https://image.tmdb.org/t/p/w780/2G249T4Sgu8gXIZpaXWnxHYYNQV.jpg', 'release_date': '2026-05-13', 'softcore': False, 'video': False, 'vote_average': 7.926, 'vote_count': 631}, {'adult': False, 'backdrop_path': 'https://image.tmdb.org/t/p/w1280/oPsRr7AfNLw6XaPuMpvkWK0bIUA.jpg', 'genre_ids': [28, 18], 'id': 1057265, 'title': 'Peddi', 'original_language': 'te', 'original_title': 'పెద్ది', 'overview': 'In 